In [1]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import time
from tqdm import tqdm

import uproot

In [2]:
parent = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/conf14"
file_path = "/mu3e_sort_run0001000.root"
in_dir = parent + file_path

#train_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/train"
#val_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/val"
#test_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/test"

train_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/train_batch_size_1"
val_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/val_batch_size_1"
test_dir = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/test_batch_size_1"

In [3]:
def is_valid_file(path):
    path = Path(path)
    return path.is_file() and path.stat().st_size > 0

In [4]:
def load_real_event_data(path_in):
    print('Loading global hit and track data, compiling events...')
    track_fields = ["tid", "pdg", "vx", "vy", "vz", "vt", "px", "py", "pz"]
    hit_fields = ["tid", "hid", "det", "pdg", "x", "y", "z", "time", "edep", "px", "py", "pz"]
        
    with uproot.open(path_in) as file:
        # pd.DataFrame of track and hit data from mu3e tree - only with chosen fields
        tracks_flat = file['mu3e_mc_tracks'].arrays(track_fields, library="pd")
        hits_flat = file['mu3e_mchits'].arrays(hit_fields, library="pd")

        # jagged awkward array - each entry in jagged array = list of hit indices in mchits tree, each list = one event
        hit_mapping = file["mu3e"]["hit_mc_i"].array(library='ak')

    # dataframe tidying    
    global_track_df = tracks_flat.rename(columns={'tid': 'trackID'})
    global_hit_df = hits_flat.rename(columns={"tid": "trackID", "hid": "hitID"})
        
    return global_track_df, global_hit_df, hit_mapping

def hit_sorter(col):
    return col.abs().astype(np.int64)

In [5]:
def root_to_parquet(
    in_dir: str, 
    train_dir: str, 
    val_dir: str, 
    test_dir: str, 
    event_batch: int,
    event_limit: int=None, 
    save_to_parquet: bool = True
):
    """
    Convert ROOT file to 2 parquet files (tracks and hits) with original eventID column.
    For hits without an associated event, assigns eventID : -1.
    """   
    path_in = Path(in_dir)
    
    path_train = Path(train_dir)
    path_val = Path(val_dir)
    path_test = Path(test_dir)
    
    path_train.mkdir(parents=True, exist_ok=True)
    path_val.mkdir(parents=True, exist_ok=True)
    path_test.mkdir(parents=True, exist_ok=True)
    
    ### loading hit and track dataframes ###
    global_track_df, global_hit_df, hit_mapping = load_real_event_data(path_in)
        
    ### mapping events ###
    print('Mapping events...')
    # flattening into one long numpy array of hit indices - all the hits in mchits that are included in monte carlo eventing
    flat_indices = np.asarray(ak.flatten(hit_mapping), dtype=np.int64)

    # creates array of eventIDs - maps onto flat_indices --- first N entries in event_ids = 0, maps to first N hit indices in flat_indices
    event_ids = np.repeat(np.arange(len(hit_mapping)), ak.num(hit_mapping))

    # above misses out all events that aren't accounted for in the mu3e tree - below assigns those events eventID = -1 -- makes easy to filter out later
    full_event_ids = np.full(len(global_hit_df), -1, dtype=np.int32)   # array of -1, length = total number of hits
    full_event_ids[flat_indices] = event_ids                           # applies eventIDs to all hit idx
    global_hit_df['eventID'] = full_event_ids                          # applied eventIDs to all hits in global hit dataframe

    tid_to_event = global_hit_df[global_hit_df['eventID'] != -1].set_index('trackID')['eventID']
    tid_to_event = tid_to_event[~tid_to_event.index.duplicated(keep='first')]
    global_track_df['eventID'] = global_track_df['trackID'].map(tid_to_event).fillna(-1).astype(int)
    
    ### applying event limit ###
    if event_limit:
        print(f'Limiting to {event_limit} compiled events...')
        global_hit_df = global_hit_df[global_hit_df["eventID"] < event_limit]
        global_track_df = global_track_df[global_track_df["eventID"] < event_limit]
        print('Ordering dataframes...')
    else:
        print('Ordering dataframes...')
        
    ### sorting eventID and trackID ###
    global_hit_df = global_hit_df.sort_values(['eventID', 'trackID'])
    global_track_df = global_track_df.sort_values(['eventID', 'trackID'])
    
    sensor_hits = global_hit_df[global_hit_df['det']==10]
    valid_sensor_hits = sensor_hits[sensor_hits['eventID']!=-1].reset_index(drop=True)
    valid_global_tracks = global_track_df[global_track_df['eventID']!=-1].reset_index(drop=True)
    
    ### dropping split tracks ###
    print('Dropping split tracks...')
    track_event_counts = valid_sensor_hits.groupby('trackID')['eventID'].nunique()
    split_tracks = track_event_counts[track_event_counts > 1].index

    valid_sensor_hits = valid_sensor_hits[~valid_sensor_hits['trackID'].isin(split_tracks)].reset_index(drop=True)
    valid_global_tracks = valid_global_tracks[~valid_global_tracks['trackID'].isin(split_tracks)].reset_index(drop=True)
    
    ### enforcing event level consistency ###
    hit_events = valid_sensor_hits['eventID'].unique()
    particle_events = valid_global_tracks['eventID'].unique()

    common_events = set(hit_events) & set(particle_events)

    valid_sensor_hits = valid_sensor_hits[valid_sensor_hits['eventID'].isin(common_events)]
    valid_global_tracks = valid_global_tracks[valid_global_tracks['eventID'].isin(common_events)]
    
    ### restricting to 4-hit or more particle tracks ###
    print('Restricting to 4-hit particle tracks...')
    counts = valid_sensor_hits['trackID'].value_counts()
    keep_particle_ids = counts[counts >= 4].index.to_numpy()

    valid_sensor_hits = valid_sensor_hits[valid_sensor_hits['trackID'].isin(keep_particle_ids)].reset_index(drop=True)
    valid_global_tracks = valid_global_tracks[valid_global_tracks['trackID'].isin(keep_particle_ids)].reset_index(drop=True)
    
    # sanity check
    bad = valid_sensor_hits.groupby('trackID').size()
    bad = bad[bad < 4]

    assert bad.empty, f"Non-4-hit tracks found: {bad.head()}"
    
    ### sorting by hitID within track ###
    print('Sorting hitID within tracks...')
    valid_sensor_hits = valid_sensor_hits.sort_values(by=['trackID', 'hitID'], key=hit_sorter)
    
    ### merge events into larger batch ###
    print(f'Merging events into batch of {event_batch}...')
    valid_sensor_hits['eventID_old'] = valid_sensor_hits['eventID']
    valid_global_tracks['eventID_old'] = valid_global_tracks['eventID']
    
    valid_sensor_hits['eventID'] = valid_sensor_hits['eventID_old'] // event_batch
    valid_global_tracks['eventID'] = valid_global_tracks['eventID_old'] // event_batch

    valid_global_tracks = valid_global_tracks.sort_values(
        ["eventID", "trackID"]
    ).reset_index(drop=True)

    ### sanity checks ###
    assert valid_sensor_hits.groupby("trackID")["eventID"].nunique().max() == 1
    assert set(valid_sensor_hits["eventID"].unique()) == set(
        valid_global_tracks["eventID"].unique()
    )
    
    print('Separating into train, val, test groups...')
    total_events = valid_sensor_hits['eventID'].max()
    train_hits = valid_sensor_hits[valid_sensor_hits["eventID"] < int(0.75*total_events)]
    train_tracks = valid_global_tracks[valid_global_tracks["eventID"] < int(0.75*total_events)]
    
    val_hits = valid_sensor_hits[(valid_sensor_hits["eventID"] >= int(0.75*total_events)) & (valid_sensor_hits["eventID"] < int(0.875*total_events))]
    val_tracks = valid_global_tracks[(valid_global_tracks["eventID"] >= int(0.75*total_events)) & (valid_global_tracks["eventID"] < int(0.875*total_events))]
    
    test_hits = valid_sensor_hits[valid_sensor_hits["eventID"] >= int(0.875*total_events)]
    test_tracks = valid_global_tracks[valid_global_tracks["eventID"] >= int(0.875*total_events)]

    ### saving to single parquet files ###
    if save_to_parquet:
        print("Saving hits to parquet...")
        train_hits.to_parquet(path_train / "all_hits.parquet", index=False)
        val_hits.to_parquet(path_val / "all_hits.parquet", index=False)    
        test_hits.to_parquet(path_test / "all_hits.parquet", index=False)

        print("Saving tracks to parquet...")
        train_tracks.to_parquet(path_train / "all_tracks.parquet", index=False)
        val_tracks.to_parquet(path_val / "all_tracks.parquet", index=False)    
        test_tracks.to_parquet(path_test / "all_tracks.parquet", index=False)

        print("\n--- Done ---")
        print(f"Saved {len(train_tracks['eventID'].unique())} events to:")
        print(f"  - {path_train / 'all_tracks.parquet'}")
        print(f"  - {path_train / 'all_hits.parquet'}")
        print()
        print(f"Saved {len(val_tracks['eventID'].unique())} events to:")
        print(f"  - {path_val / 'all_tracks.parquet'}")
        print(f"  - {path_val / 'all_hits.parquet'}")
        print()
        print(f"Saved {len(test_tracks['eventID'].unique())} events to:")
        print(f"  - {path_test / 'all_tracks.parquet'}")
        print(f"  - {path_test / 'all_hits.parquet'}")
    else:
        print("\n--- Done ---")
        print(f"Saved {len(valid_sensor_hits['eventID'].unique())} events to:")
        print(f"  - valid_sensor_hits")
        print(f"  - valid_global_tracks")
        print()
    
    return global_track_df, global_hit_df, valid_sensor_hits, valid_global_tracks

In [8]:
global_tracks, global_hits, hits, particles = root_to_parquet(in_dir, train_dir, val_dir, test_dir, event_batch = 2, event_limit=None, save_to_parquet=False)

Loading global hit and track data, compiling events...
Mapping events...
Ordering dataframes...
Dropping split tracks...
Restricting to 4-hit particle tracks...
Sorting hitID within tracks...
Merging events into batch of 2...
Separating into train, val, test groups...

--- Done ---
Saved 25000 events to:
  - valid_sensor_hits
  - valid_global_tracks



### `TIME PROCESSING`

In [23]:
event2_hits = hits[hits['eventID']==4].drop(columns=['eventID_old','det','pdg'])

In [27]:
print(event2_hits['trackID'].unique())

[12811 15622 17027 17072 19703]


In [30]:
t0 = event2_hits['time'].min()
event2_hits['raw_time'] = event2_hits['time']
event2_hits['time'] = event2_hits['time'] - t0

In [31]:
event2_hits

,trackID,hitID,x,y,z,time,edep,px,py,pz,eventID,raw_time
510,12811,1,-15.498483,18.548941,-20.447204,0.000000,0.014432,-20.838611,35.859626,-0.140370,4,8455.085814
514,12811,2,-18.318958,23.238556,-20.495195,0.018258,0.016336,-22.104258,35.056497,-0.326554,4,8455.104072
517,12811,3,-45.903372,55.531118,-20.923126,0.160452,0.017295,-31.000092,27.210680,-0.839503,4,8455.246267
521,12811,4,-56.362701,63.845452,-21.322381,0.205061,0.024569,-33.399436,24.122145,-1.221092,4,8455.290876
523,12811,-6,-7.923405,-84.715534,-43.776923,2.447104,0.021896,11.767036,38.178633,-1.817555,4,8457.532919
519,12811,-7,-4.871621,-72.050528,-44.589262,2.490665,0.010648,7.137626,39.217725,-2.647960,4,8457.576479
515,12811,-8,-3.535521,-29.638000,-47.817014,2.633190,0.015891,-4.980645,39.397043,-3.036098,4,8457.719005
512,12811,-9,-4.439750,-23.205386,-48.087549,2.654881,0.029205,-6.480253,39.231937,-1.634120,4,8457.740695
511,12811,10,-18.168369,16.417266,-49.442567,2.795433,0.029672,-19.080767,34.794429,-1.235012,4,8457.881248
509,12811,11,-18.631922,17.251028,-49.477140,2.798618,0.015945,-19.204943,34.664916,-1.579014,4,8457.884432


In [19]:
counts = hits['trackID'].value_counts()

In [20]:
hits[hits['trackID'] == 14719995]

,trackID,hitID,det,pdg,x,y,z,time,edep,px,py,pz,eventID,eventID_old
1122063,14719995,-223,10,-11,-14.438591,-70.791013,74.143658,5.222400e+06,0.063911,6.020492,1.146309,1.719592,10187,20374
1122064,14719995,-218,10,-11,10.588481,-71.297889,84.403214,5.222399e+06,0.021937,4.862769,8.708685,1.252886,10187,20374
1122065,14719995,-217,10,-11,-5.750483,-84.960363,81.787064,5.222398e+06,0.047360,9.413828,3.622783,1.119287,10187,20374
1122066,14719995,-212,10,-11,2.502787,-29.638000,55.713275,5.222398e+06,0.026701,-8.731861,5.866627,1.893779,10187,20374
1122067,14719995,-211,10,-11,15.320974,-70.674845,47.541356,5.222398e+06,0.019011,3.814876,10.069296,0.991152,10187,20374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1122257,14719995,213,10,-11,-18.024152,-23.452745,60.254575,5.222398e+06,0.020289,-10.325823,-0.210930,2.137348,10187,20374
1122258,14719995,214,10,-11,-50.108496,-52.304412,70.567766,5.222398e+06,0.021826,-1.021848,-10.203067,1.848990,10187,20374
1122259,14719995,216,10,-11,-46.217433,-71.346542,74.758373,5.222398e+06,0.045322,4.624461,-8.881653,2.222174,10187,20374
1122260,14719995,219,10,-11,-46.271178,-55.248890,80.215687,5.222399e+06,0.041397,1.281331,-7.338712,-1.416245,10187,20374


### `four-hit and event consistency`

In [58]:
# restricting to 4-hit particle tracks
hits_copy = hits.copy()
particles_copy = particles.copy()

counts = hits_copy['trackID'].value_counts()
keep_particle_ids = counts[counts == 4].index.to_numpy()

hits_copy = hits_copy[hits_copy['trackID'].isin(keep_particle_ids)]
particles_copy = particles_copy[particles_copy['trackID'].isin(keep_particle_ids)]

In [63]:
# event-level consistency
hits_copy2 = hits.copy()
particles_copy2 = particles.copy()

hit_events = hits_copy['eventID'].unique()
particle_events = particles_copy['eventID'].unique()

common_events = set(hit_events) & set(particle_events)

hits_copy = hits_copy[hits_copy['eventID'].isin(common_events)].reset_index(drop=True)
particles_copy = particles_copy[particles_copy['eventID'].isin(common_events)].reset_index(drop=True)

### `dataloader testing`

In [132]:
sample_ids = hits['eventID'].unique()

In [133]:
for sample_id in tqdm(sample_ids):
    hits_event = hits[hits['eventID'] == sample_id].copy()
    particles_event = particles[particles['eventID'] == sample_id].copy()

    # Apply particle cut based on hit content
    counts = hits_event["trackID"].value_counts()
    keep_particle_ids = counts[counts >= 4].index.to_numpy()
    particles_event = particles_event[particles_event["trackID"].isin(keep_particle_ids)]

    # Re-index tracks per event
    particles_event["particle_idx"] = np.arange(len(particles_event))
    trackID_to_idx = dict(zip(particles_event['trackID'].values, particles_event['particle_idx'].values))
    assert len(trackID_to_idx) == len(particles_event), f"event {sample_id} : trackID to particle_idx mapping is not one-to-one"

    hits_event["particle_idx"] = hits_event["trackID"].map(trackID_to_idx)
    hits_event = hits_event.dropna(subset=['particle_idx'])
    hits_event['particle_idx'] = hits_event['particle_idx'].astype(int)
    assert hits_event["particle_idx"].min() >= 0, f"event {sample_id} : particle_idx to hit mapping failed"
    assert hits_event["particle_idx"].max() < len(particles_event), f"event {sample_id} : particle_idx to hit mapping failed"

    hits_event["on_valid_particle"] = hits_event["particle_idx"].isin(particles_event["particle_idx"])

    # Sanity checks
    assert len(particles_event) != 0, f"event {sample_id} : No particles remaining - loosen selection!"
    assert len(hits_event) != 0, f"event {sample_id} : No hits remaining - loosen selection!"
    assert particles_event["trackID"].nunique() == len(particles_event), f"event {sample_id} : Non-unique particle ids"

100%|██████████| 47040/47040 [05:48<00:00, 135.14it/s]


In [122]:
bad = hits.groupby('trackID').size()
print(bad[bad != 4])
print()
hits[hits['trackID'] == 468552]

Series([], dtype: int64)



,trackID,hitID,det,pdg,x,y,z,time,edep,px,py,pz,eventID
7352,468552,1,10,-11,-18.890055,-14.674962,-55.775331,173823.979766,0.037848,-29.876010,-17.958127,-28.240827,653
7373,468552,2,10,-11,-25.058007,-18.563443,-61.680109,173824.011067,0.029896,-29.031926,-19.680680,-27.884419,654
7374,468552,3,10,-11,-54.544226,-47.189514,-95.224133,173824.188635,0.057849,-20.038371,-28.341702,-27.993357,654
7375,468552,4,10,-11,-62.422697,-57.785456,-105.966774,173824.245436,0.030251,-18.922665,-28.837373,-28.094521,654


In [128]:
# how many distinct events each trackID appears in
track_event_counts = (
    hits
    .groupby("trackID")["eventID"]
    .nunique()
)

split_tracks = track_event_counts[track_event_counts > 1]

print("Number of split tracks:", len(split_tracks))
print("Example split trackIDs:")
print(split_tracks.head())

Number of split tracks: 187
Example split trackIDs:
trackID
128606    2
230201    2
468552    2
630948    2
757662    2
Name: eventID, dtype: int64


In [129]:
len(particles)

142108

### `detector filtering`

In [7]:
event_counts = global_hit_df.groupby("eventID").size()
valid_event_ids = event_counts[event_counts > 0].index
global_hit_df_2 = global_hit_df[global_hit_df["eventID"].isin(valid_event_ids)]

In [8]:
# global hit dataframe - all hits w/ eventID = -1 are removed
valid_global_hits = global_hit_df[global_hit_df['eventID']!=-1]

In [9]:
# all sensor hits w/ eventID = -1 are removed
valid_sensor_hits = sensor_hits[sensor_hits['eventID']!=-1]

# unique track IDs froms valid sensor hits 
valid_sensor_hits_track_ids = valid_sensor_hits['trackID'].unique()

# unique track IDs from ALL sensor hits, including eventID = -1
sensor_hits_track_ids = sensor_hits['trackID'].unique()

NameError: name 'sensor_hits' is not defined

In [ ]:
# global track dataframe - all tracks w/ eventID = -1] are removed
valid_global_tracks = global_track_df[global_track_df['eventID']!=-1]
# sensor track dataframe - all tracks corresponding to hits left over in hits after filtering out all det != 10
valid_sensor_tracks = global_track_df[global_track_df['trackID'].isin(sensor_hits_track_ids)]

In [ ]:
valid_sensor_tracks_NaN = valid_sensor_tracks[valid_sensor_tracks['eventID']==-1]
valid_global_tracks_NaN = valid_global_tracks[valid_global_tracks['eventID']==-1]

In [ ]:
print(f'valid sensor tracks has {len(valid_sensor_tracks)} tracks in it')
print(f'valid global tracks has {len(valid_global_tracks)} tracks in it')
print()
print(f'valid NaN sensor tracks has {len(valid_sensor_tracks_NaN)} tracks in it')
print(f'valid NaN global tracks has {len(valid_global_tracks_NaN)} tracks in it')
print()
print('where the freak is this discrepancy coming from...')
print('just gonna get rid of all eventID = -1 for now, only a 1000 tracks out of 380k')

In [15]:
print(f"number of tracks that are assigned NaN eventID and are made up of sensor hits : {len(sensor_hits[sensor_hits['eventID']==-1]['trackID'].unique())}")

number of tracks that are assigned NaN eventID and are made up of sensor hits : 6762


### `enforcing num_particles == 4`

In [10]:
print(f"number of events in hits {len(valid_sensor_hits['eventID'].unique())}")
print(f"number of events in particles {len(valid_global_tracks['eventID'].unique())}")

number of events in hits 49975
number of events in particles 49973


In [22]:
hits = valid_sensor_hits.copy()
particles = valid_global_tracks.copy()

counts = hits['trackID'].value_counts()
keep_particle_ids = counts[counts == 4].index.to_numpy()

hits = hits[hits['trackID'].isin(keep_particle_ids)]
particles = particles[particles['trackID'].isin(keep_particle_ids)]

In [23]:
hit_events = set(hits['eventID'].unique())
particle_events = set(particles['eventID'].unique())

common = hit_events & particle_events

hits = hits['eventID'].isin(common)
particles = particles['eventID'].isin(common)

In [24]:
print(f"number of events in hits {len(hit_eventIDs)}")
print(f"number of events in particles {len(particle_eventIDs)}")

NameError: name 'hit_eventIDs' is not defined

ok need to make sure eventIDs in hits and particles are matching so same amount of hits and events - THIS MIGHT BE MATCHER ERROR OH MY GOD

In [14]:
len(common_eventIDs)

47054

In [17]:
uncommon_eventIDs = set(hits['eventID']) ^ set(particles['eventID'])

In [18]:
uncommon_eventIDs

{654, 9273, 24005, 24250, 27358, 37274, 40747, 44600, 48713}

## `EDA`

In [31]:
hit_df = valid_sensor_hits
track_df = valid_sensor_tracks

print(f"unique particle ids:   {np.unique(hit_df['pdg'].values).tolist()}")
print("")
print("--------------------------- hit stats ---------------------------")
print(f"electron hits:                  {(hit_df['pdg'].values == 11).sum()}")
print(f"positron hits:                  {(hit_df['pdg'].values == -11).sum()}")
print(f"antimuon hits:                  {(hit_df['pdg'].values == -13).sum()}")
print(f"photon hits:                    {(hit_df['pdg'].values == 22).sum()}")
print(f"proton hits:                    {(hit_df['pdg'].values == 2212).sum()}")
print(f"neutron hits:                   {(hit_df['pdg'].values == 2112).sum()}")
print(f"alpha particle hits:            {(hit_df['pdg'].values == 1000020040).sum()}")
print(f"Be nuclide hits:                {(hit_df['pdg'].values == 1000040080).sum()}")
print(f"C nuclide hits:                 {(hit_df['pdg'].values == 1000060120).sum()}")
print(f"Mg nuclide hits:                {(hit_df['pdg'].values == 1000120240).sum()}")
print(f"Si nuclide hits:                {(hit_df['pdg'].values == 1000140280).sum()}")
print(f"hits from unassigned events:    {(hit_df['eventID'].values == -1).sum()}")
print("")
print(f"total hits:                     {len(hit_df)}")
print("")
print("-------------------------- track stats --------------------------")
print(f"electron tracks:                {(track_df['pdg'].values == 11).sum()}")
print(f"positron tracks:                {(track_df['pdg'].values == -11).sum()}")
print(f"antimuon tracks:                {(track_df['pdg'].values == -13).sum()}")
print(f"photon tracks:                  {(track_df['pdg'].values == 22).sum()}")
print(f"proton tracks:                  {(track_df['pdg'].values == 2212).sum()}")
print(f"neutron tracks:                 {(track_df['pdg'].values == 2112).sum()}")
print(f"alpha particle tracks:          {(track_df['pdg'].values == 1000020040).sum()}")
print(f"Be nuclide tracks:              {(track_df['pdg'].values == 1000040080).sum()}")
print(f"C nuclide tracks:               {(track_df['pdg'].values == 1000060120).sum()}")
print(f"Mg nuclide tracks:              {(track_df['pdg'].values == 1000120240).sum()}")
print(f"Si nuclide tracks:              {(track_df['pdg'].values == 1000140280).sum()}")
print(f"tracks from unassigned events:  {(track_df['eventID'].values == -1).sum()}")
print("")
print(f"total tracks:                   {len(track_df)}")

unique particle ids:   [-13, -11, 11, 1000120240, 1000140280]

--------------------------- hit stats ---------------------------
electron hits:                  117792
positron hits:                  2860066
antimuon hits:                  87
photon hits:                    0
proton hits:                    0
neutron hits:                   0
alpha particle hits:            0
Be nuclide hits:                0
C nuclide hits:                 0
Mg nuclide hits:                1
Si nuclide hits:                1
hits from unassigned events:    0

total hits:                     2977947

-------------------------- track stats --------------------------
electron tracks:                18479
positron tracks:                362183
antimuon tracks:                82
photon tracks:                  0
proton tracks:                  0
neutron tracks:                 0
alpha particle tracks:          1
Be nuclide tracks:              0
C nuclide tracks:               0
Mg nuclide tracks:         

## `PLOTTING`

In [ ]:
def plot_event(event):
    """
    Plot hit positions for a single event.
    
    Parameters:
    -----------
    event : awkward array
        Single event data containing hit position information
    """
    # Extract hit positions (adjust field names based on your data structure)
    x = ak.concatenate([event[key]['x'] for key in event.keys()])
    y = ak.concatenate([event[key]['y'] for key in event.keys()])
    z = ak.concatenate([event[key]['z'] for key in event.keys()])
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 3, figsize=(20,6.5))
    
    # XY projection
    axes[0].scatter(x, y, alpha=0.8, s=10)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    axes[0].set_title('XY Projection')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-100,100)
    axes[0].set_ylim(-100,100)
    
    # XZ projection
    axes[1].scatter(z, x, alpha=0.8, s=10)
    axes[1].set_xlabel('z')
    axes[1].set_ylabel('x')
    axes[1].set_title('XZ Projection')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(-100,100)

    
    # YZ projection
    axes[2].scatter(z, y, alpha=0.8, s=10)
    axes[2].set_xlabel('z')
    axes[2].set_ylabel('y')
    axes[2].set_title('YZ Projection')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim(-100,100)

    
    plt.tight_layout()
    plt.show()
    
    print(f"Number of hits: {len(x)}")